# Volatility and Risk

## What you'll learn
- Measuring risk as the volatility of returns
- Rolling volatility over time
- Annualising a daily figure
- Drawdown: the worst drop from a peak

In [ ]:
import pandas as pd

df = pd.read_csv("data/stock_history.csv", parse_dates=["Date"])
df = df.drop_duplicates().sort_values("Date")
# One price column per ticker, indexed by date:
closes = df.pivot_table(index="Date", columns="Ticker", values="Close")
closes = closes.ffill()   # carry the last known price over any gaps
print(closes.head())

In [ ]:
returns = closes.pct_change().dropna()
print(returns.head())

## Volatility = spread of returns

Risk, in its simplest form, is how much returns bounce around. That is just the **standard
deviation** of the daily returns: a higher number means a bumpier ride.

In [ ]:
daily_vol = returns.std()
print(daily_vol.sort_values(ascending=False))

## Annualising

A daily figure is small and hard to interpret. Multiplying by the square root of the number
of trading days in a year (about 252) scales it to an annual figure, the conventional way
volatility is quoted.

In [ ]:
import numpy as np

annual_vol = daily_vol * np.sqrt(252)
print((annual_vol * 100).round(1))   # as a percentage

## Rolling volatility

A single number hides how risk changes over time. A rolling standard deviation shows
calm and turbulent periods.

In [ ]:
import matplotlib.pyplot as plt

rolling_vol = returns["HELIOS"].rolling(30).std() * np.sqrt(252)
ax = rolling_vol.plot(figsize=(9, 4), title="HELIOS 30-day rolling volatility (annualised)")
ax.set_ylabel("Volatility")
plt.tight_layout()
plt.show()

## Drawdown

**Drawdown** is how far an investment has fallen from its highest point so far. The biggest
such drop (the **maximum drawdown**) is a very intuitive risk measure: the worst peak-to-low
loss you would have lived through.

In [ ]:
growth = (1 + returns["ATLAS"]).cumprod()
running_peak = growth.cummax()
drawdown = (growth - running_peak) / running_peak

print("Worst drawdown for ATLAS:", round(drawdown.min() * 100, 1), "%")

In [ ]:
ax = drawdown.plot(figsize=(9, 4), title="ATLAS drawdown from peak")
ax.set_ylabel("Drawdown")
plt.tight_layout()
plt.show()

## Practice

1. Build clean wide `closes` and daily `returns`.
2. Print the annualised volatility of every ticker, sorted.
3. Plot the 30-day rolling annualised volatility of `VERTEX`.
4. Compute the maximum drawdown of `NORTH`.
5. Say which ticker is riskiest by annual volatility and which had the worst drawdown.

In [ ]:
# Your practice code here

## Summary

- Volatility = standard deviation of returns; higher means riskier.
- Annualise a daily figure by multiplying by `sqrt(252)`.
- Rolling standard deviation shows how risk changes over time.
- Drawdown is the drop from the running peak; the maximum drawdown is a key risk measure.

## Practice Solutions

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/stock_history.csv", parse_dates=["Date"]).drop_duplicates().sort_values("Date")
closes = df.pivot_table(index="Date", columns="Ticker", values="Close").ffill()
returns = closes.pct_change().dropna()

# 2.
annual_vol = returns.std() * np.sqrt(252)
print((annual_vol * 100).round(1).sort_values(ascending=False))

# 4.
growth = (1 + returns["NORTH"]).cumprod()
dd = (growth - growth.cummax()) / growth.cummax()
print("NORTH max drawdown:", round(dd.min() * 100, 1), "%")

# 5.
print("Riskiest by volatility:", annual_vol.idxmax())